In [ ]:
!pip install -q openai networkx numpy

In [ ]:
import os
import json
import hashlib
from typing import List, Dict, Tuple, Optional, Any, Set
from dataclasses import dataclass, field
from enum import Enum
import numpy as np
import networkx as nx
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "your-api-key-here"
client = OpenAI()

## Setup

In [ ]:
# Minimal document container
@dataclass
class Document:
    content: str
    doc_id: str = ""
    embedding: Optional[np.ndarray] = field(default=None, repr=False)
    metadata: Dict[str, Any] = field(default_factory=dict)
    score: float = 0.0

    def __post_init__(self):
        if not self.doc_id:
            self.doc_id = f"doc_{hashlib.md5(self.content.encode()).hexdigest()[:8]}"

    def __hash__(self):
        return hash(self.doc_id)


def embed_texts(texts: List[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """# (num_texts,) → (num_texts, embed_dim)"""
    response = client.embeddings.create(model=model, input=texts)
    emb = np.array([d.embedding for d in response.data])
    return emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-10)


def llm_generate(prompt: str, system: str = "", temperature: float = 0.7) -> str:
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    return client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=temperature
    ).choices[0].message.content


def llm_json(prompt: str, system: str = "") -> Dict[str, Any]:
    sys = (system + "\n" if system else "") + "Respond ONLY with valid JSON."
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": sys}, {"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


class SimpleVectorStore:
    """Minimal store. Use Chroma/Qdrant from Basic RAG in production."""
    def __init__(self):
        self.documents: List[Document] = []
        self._embeddings: Optional[np.ndarray] = None

    def add(self, docs: List[Document]) -> None:
        to_embed = [d.content for d in docs if d.embedding is None]
        if to_embed:
            new_emb = embed_texts(to_embed)
            j = 0
            for d in docs:
                if d.embedding is None:
                    d.embedding = new_emb[j]; j += 1
        self.documents.extend(docs)
        self._embeddings = np.vstack([d.embedding for d in self.documents])

    def search(self, query: str, k: int = 5) -> List[Document]:
        if self._embeddings is None: return []
        q_emb = embed_texts([query])[0]
        sims = q_emb @ self._embeddings.T  # (embed_dim,) @ (embed_dim, num_docs) → (num_docs,)
        k = min(k, len(self.documents))
        top_idx = np.argpartition(sims, -k)[-k:]
        top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]
        return [Document(content=self.documents[i].content, doc_id=self.documents[i].doc_id,
                         embedding=self.documents[i].embedding, metadata=self.documents[i].metadata,
                         score=float(sims[i])) for i in top_idx]

In [ ]:
KNOWLEDGE_BASE = [
    {"content": "Quantum Neural Networks (QNNs) leverage quantum superposition to represent exponentially many states. Unlike classical neural networks, QNNs explore multiple solution paths in parallel through quantum parallelism.", "entities": ["QNN", "quantum superposition"], "topic": "fundamentals"},
    {"content": "Variational Quantum Eigensolvers (VQE) are hybrid quantum-classical algorithms optimizing parameterized quantum circuits. The quantum computer evaluates cost while classical optimization updates parameters.", "entities": ["VQE", "parameterized quantum circuits"], "topic": "algorithms"},
    {"content": "The barren plateau problem occurs in variational quantum algorithms when gradients vanish exponentially with circuit depth, analogous to vanishing gradients in deep classical networks.", "entities": ["barren plateau", "vanishing gradients"], "topic": "challenges"},
    {"content": "Professor Elena Vasquez at MIT developed the Entanglement-Enhanced Gradient Descent (EEGD) algorithm in 2023. EEGD uses controlled entanglement to escape local minima, addressing the barren plateau problem.", "entities": ["Elena Vasquez", "MIT", "EEGD", "barren plateau"], "topic": "researchers"},
    {"content": "Dr. James Chen at Google Quantum AI proposed the Layerwise Learning Protocol (LLP) in 2024. LLP trains variational circuits layer-by-layer, maintaining gradient magnitudes.", "entities": ["James Chen", "Google Quantum AI", "LLP"], "topic": "researchers"},
    {"content": "QAOA solves combinatorial optimization by alternating cost and mixer Hamiltonians. Applied to MAX-CUT, portfolio optimization, and scheduling.", "entities": ["QAOA", "MAX-CUT"], "topic": "algorithms"},
    {"content": "Noise causes decoherence where quantum states lose coherence. Error mitigation techniques like zero-noise extrapolation help recover accurate expectation values.", "entities": ["decoherence", "error mitigation"], "topic": "challenges"},
    {"content": "Quantum kernel methods map classical data to quantum Hilbert spaces where inner products use quantum circuits, enabling quantum-enhanced SVMs.", "entities": ["quantum kernel", "SVM"], "topic": "methods"},
    {"content": "The Quantum Machine Learning Institute (QMLI) published benchmarks showing quantum advantage for molecular simulation. Their 2024 study compared 127-qubit processors against tensor networks.", "entities": ["QMLI", "quantum advantage", "molecular simulation"], "topic": "research_orgs"},
    {"content": "Quantum reservoir computing uses fixed random quantum dynamics as a reservoir. Only the readout layer is trained, avoiding barren plateau issues entirely.", "entities": ["quantum reservoir computing", "barren plateau"], "topic": "methods"},
    {"content": "IBM Quantum Network provides cloud access to quantum processors. Their Qiskit framework enables hybrid programming with automatic circuit optimization.", "entities": ["IBM Quantum", "Qiskit"], "topic": "platforms"},
    {"content": "Quantum ML faces the data loading bottleneck: encoding classical data requires O(N) operations. qRAM could solve this but remains experimentally challenging.", "entities": ["data loading bottleneck", "qRAM"], "topic": "challenges"},
    {"content": "Tensor networks provide efficient classical simulations of bounded-entanglement circuits. Matrix Product States (MPS) use polynomial resources when entanglement is limited.", "entities": ["tensor networks", "MPS"], "topic": "methods"},
    {"content": "Elena Vasquez collaborated with QMLI on applying EEGD to drug discovery. Their joint paper showed 40% faster convergence than standard VQE.", "entities": ["Elena Vasquez", "QMLI", "EEGD", "drug discovery", "VQE"], "topic": "applications"},
    {"content": "Google Quantum AI and IBM Quantum lead quantum hardware. Google achieved quantum supremacy in 2019; IBM focuses on error-corrected logical qubits.", "entities": ["Google Quantum AI", "IBM Quantum", "quantum supremacy"], "topic": "industry"}
]

documents = [Document(content=e["content"], doc_id=f"kb_{i:03d}",
                       metadata={"entities": e["entities"], "topic": e["topic"]})
             for i, e in enumerate(KNOWLEDGE_BASE)]
vector_store = SimpleVectorStore()
vector_store.add(documents)
print(f"Loaded {len(documents)} documents")

---

# 1) Agentic RAG Implementation

**Extends Section 4.11** from Basic RAG with full working code.

The Agentic RAG loop:
```
┌─────────────────────────────────────────────────────────────┐
│   ┌─────────┐    ┌──────────┐    ┌──────────┐    ┌───────┐ │
│   │ Retrieve │───►│ Reflect  │───►│ Decide   │───►│ Refine│ │
│   └─────────┘    └──────────┘    └──────────┘    └───────┘ │
│        ▲                                             │      │
│        └─────────────── If REFINE ───────────────────┘      │
└─────────────────────────────────────────────────────────────┘
```

## 1.1 Self-Reflection: Retrieval Quality Assessment

The agent evaluates whether retrieved documents are **sufficient** to answer the query.

In [ ]:
@dataclass
class RetrievalAssessment:
    """Structured self-reflection output."""
    is_sufficient: bool          # Can we answer with current context?
    confidence: float            # [0,1] how confident
    gaps: List[str]              # What's missing
    refinements: List[str]       # Suggested better queries
    reasoning: str               # Brief explanation


ASSESSMENT_PROMPT = """
Assess if retrieved documents sufficiently answer the question.

QUESTION: {query}

DOCUMENTS:
{documents}

Analyze: (1) Enough info to fully answer? (2) What's missing? (3) Better queries?

JSON response:
{{"is_sufficient": bool, "confidence": 0.0-1.0, "gaps": [...], "refinements": [...], "reasoning": "..."}}
"""


def assess_retrieval(query: str, docs: List[Document]) -> RetrievalAssessment:
    """
    Self-reflection on retrieval quality.

    # (query, docs) → RetrievalAssessment
    """
    docs_text = "\n\n".join(
        f"[Doc {i+1}] (score: {d.score:.3f})\n{d.content}" for i, d in enumerate(docs)
    ) if docs else "No documents retrieved."

    result = llm_json(ASSESSMENT_PROMPT.format(query=query, documents=docs_text))

    return RetrievalAssessment(
        is_sufficient=result.get("is_sufficient", False),
        confidence=result.get("confidence", 0.0),
        gaps=result.get("gaps", []),
        refinements=result.get("refinements", []),
        reasoning=result.get("reasoning", "")
    )

## 1.2 Query Refinement

When retrieval is insufficient, generate improved queries using identified gaps and discovered terminology.

In [ ]:
REFINEMENT_PROMPT = """
Generate improved search queries to fill information gaps.

ORIGINAL: {original_query}
GAPS: {gaps}
CONTEXT: {context}
PREVIOUS QUERIES: {previous_queries}

Generate 2-3 refined queries targeting gaps, using context terminology, differing from previous.

JSON: {{"refined_queries": ["query1", "query2"]}}
"""


def refine_query(original: str, gaps: List[str], context_docs: List[Document], previous: List[str]) -> List[str]:
    """
    Generate refined queries from gaps and context.

    # (original, gaps, docs, prev) → List[refined_queries]
    """
    context = "\n".join(f"- {d.content[:150]}..." for d in context_docs[:3]) if context_docs else "None"
    result = llm_json(REFINEMENT_PROMPT.format(
        original_query=original,
        gaps="\n".join(f"- {g}" for g in gaps) if gaps else "None",
        context=context,
        previous_queries="\n".join(f"- {q}" for q in previous) if previous else "None"
    ))
    return result.get("refined_queries", [original])

## 1.3 Agentic RAG Controller

Orchestrates the full loop: Retrieve → Assess → Decide → Refine/Answer

In [ ]:
class AgentAction(Enum):
    RETRIEVE = "retrieve"
    ANSWER = "answer"


@dataclass
class IterationRecord:
    iteration: int
    query: str
    docs_retrieved: int
    assessment: RetrievalAssessment
    action: AgentAction


@dataclass
class AgenticRAGResult:
    answer: str
    final_context: List[Document]
    iterations: List[IterationRecord]
    total_iterations: int

In [ ]:
class AgenticRAGController:
    """
    Full implementation of iterative retrieval with self-reflection.

    The agent autonomously decides when to:
    - Retrieve more with refined queries
    - Generate final answer from accumulated context
    """

    ANSWER_PROMPT = """Answer based on context. Acknowledge limitations if insufficient.

CONTEXT:
{context}

QUESTION: {query}
"""

    def __init__(self, vector_store: SimpleVectorStore, max_iterations: int = 3,
                 docs_per_iter: int = 3, confidence_threshold: float = 0.7):
        self.store = vector_store
        self.max_iterations = max_iterations
        self.docs_per_iter = docs_per_iter
        self.confidence_threshold = confidence_threshold

    def _should_continue(self, assessment: RetrievalAssessment, iteration: int) -> bool:
        """# (assessment, iter) → bool: True = continue iterating"""
        if iteration >= self.max_iterations:
            return False
        if assessment.is_sufficient and assessment.confidence >= self.confidence_threshold:
            return False
        if not assessment.refinements:
            return False
        return True

    def _deduplicate(self, docs: List[Document]) -> List[Document]:
        """# (docs) → unique docs sorted by score desc"""
        seen: Dict[str, Document] = {}
        for doc in docs:
            if doc.doc_id not in seen or doc.score > seen[doc.doc_id].score:
                seen[doc.doc_id] = doc
        return sorted(seen.values(), key=lambda d: d.score, reverse=True)

    def query(self, user_query: str, verbose: bool = True) -> AgenticRAGResult:
        """# (query) → AgenticRAGResult"""
        accumulated: List[Document] = []
        previous_queries: List[str] = []
        iterations: List[IterationRecord] = []
        current_query = user_query

        for iter_num in range(self.max_iterations):
            if verbose:
                print(f"\n{'='*50}\nIteration {iter_num+1}: '{current_query}'")

            # Retrieve
            retrieved = self.store.search(current_query, k=self.docs_per_iter)
            if verbose:
                print(f"Retrieved {len(retrieved)} docs")
                for d in retrieved:
                    print(f"  [{d.doc_id}] {d.score:.3f}: {d.content[:40]}...")

            # Accumulate & deduplicate
            accumulated.extend(retrieved)
            accumulated = self._deduplicate(accumulated)

            # Assess against ORIGINAL query
            assessment = assess_retrieval(user_query, accumulated)
            if verbose:
                print(f"\nAssessment: sufficient={assessment.is_sufficient}, conf={assessment.confidence:.2f}")
                print(f"  Reasoning: {assessment.reasoning}")
                if assessment.gaps:
                    print(f"  Gaps: {assessment.gaps}")

            previous_queries.append(current_query)

            # Decide
            should_continue = self._should_continue(assessment, iter_num + 1)
            action = AgentAction.RETRIEVE if should_continue else AgentAction.ANSWER

            iterations.append(IterationRecord(
                iteration=iter_num + 1, query=current_query,
                docs_retrieved=len(retrieved), assessment=assessment, action=action
            ))

            if action == AgentAction.ANSWER:
                if verbose:
                    print("\nAction: ANSWER")
                break

            # Refine
            refined = refine_query(user_query, assessment.gaps, accumulated, previous_queries)
            if refined:
                current_query = refined[0]
                if verbose:
                    print(f"\nAction: RETRIEVE with '{current_query}'")
            else:
                break

        # Generate answer
        if verbose:
            print(f"\n{'='*50}\nGenerating answer with {len(accumulated)} docs")

        context_text = "\n\n".join(f"[{i+1}] {d.content}" for i, d in enumerate(accumulated))
        answer = llm_generate(
            self.ANSWER_PROMPT.format(context=context_text, query=user_query),
            temperature=0.3
        )

        return AgenticRAGResult(
            answer=answer, final_context=accumulated,
            iterations=iterations, total_iterations=len(iterations)
        )

## 1.4 Agentic RAG Demo

In [ ]:
agentic_rag = AgenticRAGController(vector_store, max_iterations=3, docs_per_iter=3)

# Query requiring iteration: initial search finds problem, refinement finds solutions
result = agentic_rag.query("What approaches address training difficulties in quantum neural networks?")

print("\n" + "="*70 + "\nFINAL ANSWER:\n" + "="*70)
print(result.answer)
print(f"\nTotal iterations: {result.total_iterations}")

---

# 2) Graph RAG Implementation

```
Query ──► Entity    ──► Graph      ──► Document  ──► Answer
         Linking       Traversal      Retrieval
```

## 2.1 Entity and Relation Data Structures

In [ ]:
@dataclass
class Entity:
    """Named entity with type and source tracking."""
    name: str
    entity_type: str              # PERSON, ORG, ALGORITHM, CONCEPT, PLATFORM
    normalized: str = ""          # Lowercase for matching
    source_docs: Set[str] = field(default_factory=set)

    def __post_init__(self):
        if not self.normalized:
            self.normalized = self.name.lower().strip()

    def __hash__(self):
        return hash(self.normalized)

    def __eq__(self, other):
        return isinstance(other, Entity) and self.normalized == other.normalized


@dataclass
class Relation:
    """Directed edge between entities."""
    source: str                   # Normalized source entity
    relation_type: str            # DEVELOPED, AFFILIATED_WITH, ADDRESSES, etc.
    target: str                   # Normalized target entity
    source_doc: str = ""

    def __hash__(self):
        return hash((self.source, self.relation_type, self.target))

    def __eq__(self, other):
        return (isinstance(other, Relation) and self.source == other.source
                and self.relation_type == other.relation_type and self.target == other.target)

## 2.2 Entity-Relation Extraction

LLM extracts structured knowledge from unstructured text.

In [ ]:
EXTRACTION_PROMPT = """
Extract entities and relationships.

ENTITY TYPES: PERSON, ORGANIZATION, ALGORITHM, CONCEPT, PLATFORM
RELATION TYPES: DEVELOPED, AFFILIATED_WITH, ADDRESSES, USES, COLLABORATES, PUBLISHED

TEXT: {text}

JSON:
{{
    "entities": [{{"name": "...", "type": "..."}}],
    "relations": [{{"source": "...", "relation": "...", "target": "..."}}]
}}
"""


def extract_entities_relations(doc: Document) -> Tuple[List[Entity], List[Relation]]:
    """# (Document) → (List[Entity], List[Relation])"""
    try:
        result = llm_json(EXTRACTION_PROMPT.format(text=doc.content))
    except:
        return [], []

    entities = [Entity(name=e.get("name", ""), entity_type=e.get("type", "CONCEPT"),
                       source_docs={doc.doc_id}) for e in result.get("entities", [])]

    relations = [Relation(source=r.get("source", "").lower().strip(),
                          relation_type=r.get("relation", "RELATED_TO"),
                          target=r.get("target", "").lower().strip(),
                          source_doc=doc.doc_id) for r in result.get("relations", [])]

    return entities, relations

## 2.3 Knowledge Graph

NetworkX-based graph with entity lookup, neighbor traversal, and path finding.

In [ ]:
class KnowledgeGraph:
    """Knowledge graph with entity/relation management and traversal."""

    def __init__(self):
        self.graph = nx.DiGraph()
        self.entities: Dict[str, Entity] = {}           # normalized → Entity
        self.relations: Set[Relation] = set()
        self.entity_to_docs: Dict[str, Set[str]] = {}   # normalized → doc_ids

    def add_entity(self, entity: Entity) -> None:
        norm = entity.normalized
        if norm in self.entities:
            self.entities[norm].source_docs.update(entity.source_docs)
        else:
            self.entities[norm] = entity
            self.graph.add_node(norm, name=entity.name, entity_type=entity.entity_type)
        self.entity_to_docs.setdefault(norm, set()).update(entity.source_docs)

    def add_relation(self, relation: Relation) -> None:
        for n in [relation.source, relation.target]:
            if n not in self.graph:
                self.graph.add_node(n)
        self.graph.add_edge(relation.source, relation.target,
                            relation_type=relation.relation_type, source_doc=relation.source_doc)
        self.relations.add(relation)

    def get_neighbors(self, entity_name: str, max_hops: int = 1,
                      relation_filter: Optional[List[str]] = None) -> List[Tuple[str, str, str]]:
        """# (entity, hops, filter) → List[(src, rel, tgt)]"""
        norm = entity_name.lower().strip()
        if norm not in self.graph:
            return []

        results, visited, frontier = [], {norm}, [norm]

        for _ in range(max_hops):
            next_frontier = []
            for node in frontier:
                # Outgoing
                for _, target, data in self.graph.out_edges(node, data=True):
                    rel = data.get("relation_type", "RELATED_TO")
                    if relation_filter and rel not in relation_filter:
                        continue
                    results.append((node, rel, target))
                    if target not in visited:
                        visited.add(target)
                        next_frontier.append(target)
                # Incoming
                for source, _, data in self.graph.in_edges(node, data=True):
                    rel = data.get("relation_type", "RELATED_TO")
                    if relation_filter and rel not in relation_filter:
                        continue
                    results.append((source, rel, node))
                    if source not in visited:
                        visited.add(source)
                        next_frontier.append(source)
            frontier = next_frontier

        return list(set(results))

    def get_entity_docs(self, entity_name: str) -> Set[str]:
        return self.entity_to_docs.get(entity_name.lower().strip(), set())

    def get_entities_by_type(self, entity_type: str) -> List[Entity]:
        return [e for e in self.entities.values() if e.entity_type.upper() == entity_type.upper()]

    def find_paths(self, source: str, target: str, max_length: int = 3) -> List[List[str]]:
        src, tgt = source.lower().strip(), target.lower().strip()
        if src not in self.graph or tgt not in self.graph:
            return []
        try:
            return list(nx.all_simple_paths(self.graph.to_undirected(), src, tgt, cutoff=max_length))
        except nx.NetworkXNoPath:
            return []

## 2.4 Entity Linker

Links query mentions to graph entities using exact matching + embedding similarity.

In [ ]:
class EntityLinker:
    """Links query to graph entities via exact + embedding similarity."""

    def __init__(self, kg: KnowledgeGraph, similarity_threshold: float = 0.65):
        self.kg = kg
        self.threshold = similarity_threshold
        self._entity_names: List[str] = []
        self._entity_embeddings: Optional[np.ndarray] = None  # (num_entities, embed_dim)

    def build_index(self) -> None:
        self._entity_names = list(self.kg.entities.keys())
        if self._entity_names:
            self._entity_embeddings = embed_texts(self._entity_names)

    def link(self, query: str, top_k: int = 5) -> List[Tuple[Entity, float]]:
        """# (query, k) → List[(Entity, score)]"""
        if not self._entity_names:
            return []

        matched = []
        query_lower = query.lower()

        # Exact substring matching
        for name, entity in self.kg.entities.items():
            if name in query_lower or query_lower in name:
                matched.append((entity, 1.0))

        # Embedding similarity
        q_emb = embed_texts([query])[0]  # (embed_dim,)
        sims = q_emb @ self._entity_embeddings.T  # (num_entities,)

        k_actual = min(top_k, len(self._entity_names))
        top_idx = np.argpartition(sims, -k_actual)[-k_actual:]
        top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]

        for idx in top_idx:
            if sims[idx] >= self.threshold:
                entity = self.kg.entities[self._entity_names[idx]]
                if not any(e.normalized == entity.normalized for e, _ in matched):
                    matched.append((entity, float(sims[idx])))

        matched.sort(key=lambda x: x[1], reverse=True)
        return matched[:top_k]

## 2.5 Graph RAG Retriever

Hybrid retrieval combining graph structure with vector similarity:

$$\text{hybrid_score} = \alpha \cdot \text{vector_sim} + (1 - \alpha) \cdot \text{graph_presence}$$

In [ ]:
@dataclass
class GraphRAGResult:
    documents: List[Document]
    query_entities: List[Entity]
    traversed_relations: List[Tuple[str, str, str]]
    answer: str = ""


class GraphRAGRetriever:
    """Hybrid graph + vector retrieval."""

    ANSWER_PROMPT = """Answer using entities, relationships, and documents.

ENTITIES: {entities}
RELATIONSHIPS: {relations}
DOCUMENTS: {documents}

QUESTION: {query}
"""

    def __init__(self, kg: KnowledgeGraph, vector_store: SimpleVectorStore,
                 linker: EntityLinker, traversal_hops: int = 2, alpha: float = 0.5):
        self.kg = kg
        self.store = vector_store
        self.linker = linker
        self.hops = traversal_hops
        self.alpha = alpha  # Balance: 1.0 = vector only, 0.0 = graph only

    def _hybrid_rank(self, graph_doc_ids: Set[str], vector_results: List[Document]) -> List[Document]:
        """# (graph_docs, vector_docs) → hybrid ranked docs"""
        all_docs: Dict[str, Document] = {}
        graph_scores: Dict[str, float] = {}
        vector_scores: Dict[str, float] = {}

        # Graph docs get score 1.0
        for doc in self.store.documents:
            if doc.doc_id in graph_doc_ids:
                all_docs[doc.doc_id] = doc
                graph_scores[doc.doc_id] = 1.0

        # Vector results
        for doc in vector_results:
            all_docs[doc.doc_id] = doc
            vector_scores[doc.doc_id] = doc.score
            graph_scores.setdefault(doc.doc_id, 0.0)

        for doc_id in all_docs:
            vector_scores.setdefault(doc_id, 0.0)

        # hybrid = α * vector + (1-α) * graph
        hybrid = {doc_id: self.alpha * vector_scores[doc_id] + (1 - self.alpha) * graph_scores[doc_id]
                  for doc_id in all_docs}

        sorted_ids = sorted(hybrid.keys(), key=lambda x: hybrid[x], reverse=True)
        return [Document(content=all_docs[d].content, doc_id=d, embedding=all_docs[d].embedding,
                         metadata=all_docs[d].metadata, score=hybrid[d]) for d in sorted_ids]

    def retrieve(self, query: str, top_k: int = 5, verbose: bool = True) -> GraphRAGResult:
        if verbose:
            print(f"\nGraph RAG: '{query}'\n" + "="*50)

        # Link entities
        linked = self.linker.link(query, top_k=5)
        query_entities = [e for e, _ in linked]

        if verbose:
            print("Linked Entities:")
            for e, s in linked:
                print(f"  - {e.name} ({e.entity_type}): {s:.3f}")

        # Graph traversal
        all_relations = []
        expanded_entities = set(query_entities)

        for entity in query_entities:
            relations = self.kg.get_neighbors(entity.normalized, max_hops=self.hops)
            all_relations.extend(relations)
            for src, _, tgt in relations:
                if src in self.kg.entities:
                    expanded_entities.add(self.kg.entities[src])
                if tgt in self.kg.entities:
                    expanded_entities.add(self.kg.entities[tgt])

        unique_relations = list(set(all_relations))

        if verbose:
            print(f"\nTraversal: {len(expanded_entities)} entities, {len(unique_relations)} relations")
            for s, r, t in unique_relations[:5]:
                print(f"  {s} --[{r}]--> {t}")

        # Collect graph docs
        graph_doc_ids = set()
        for entity in expanded_entities:
            graph_doc_ids.update(self.kg.get_entity_docs(entity.normalized))

        # Vector search
        vector_results = self.store.search(query, k=top_k)

        if verbose:
            print(f"\nDocs from graph: {len(graph_doc_ids)}, from vector: {len(vector_results)}")

        # Hybrid rank
        ranked = self._hybrid_rank(graph_doc_ids, vector_results)[:top_k]

        if verbose:
            print("\nHybrid Ranked:")
            for d in ranked:
                print(f"  [{d.doc_id}] {d.score:.3f}: {d.content[:40]}...")

        return GraphRAGResult(documents=ranked, query_entities=query_entities,
                              traversed_relations=unique_relations)

    def query(self, user_query: str, top_k: int = 5, verbose: bool = True) -> GraphRAGResult:
        result = self.retrieve(user_query, top_k, verbose)

        entities_text = "\n".join(f"- {e.name} ({e.entity_type})" for e in result.query_entities) or "None"
        relations_text = "\n".join(f"- {s} --[{r}]--> {t}" for s, r, t in result.traversed_relations[:10]) or "None"
        docs_text = "\n\n".join(f"[{i+1}] {d.content}" for i, d in enumerate(result.documents)) or "None"

        result.answer = llm_generate(
            self.ANSWER_PROMPT.format(entities=entities_text, relations=relations_text,
                                      documents=docs_text, query=user_query),
            temperature=0.3
        )
        return result

## 2.6 Build Knowledge Graph

In [ ]:
kg = KnowledgeGraph()

print("Extracting entities and relations...\n")
for doc in documents:
    entities, relations = extract_entities_relations(doc)
    print(f"{doc.doc_id}: {len(entities)} entities, {len(relations)} relations")
    for e in entities:
        kg.add_entity(e)
    for r in relations:
        kg.add_relation(r)

print(f"\nKnowledge Graph: {len(kg.entities)} entities, {len(kg.relations)} relations")
print(f"NetworkX: {kg.graph.number_of_nodes()} nodes, {kg.graph.number_of_edges()} edges")

In [ ]:
# Initialize linker and retriever
linker = EntityLinker(kg, similarity_threshold=0.6)
linker.build_index()
print(f"Entity linker: {len(linker._entity_names)} entities indexed")

graph_rag = GraphRAGRetriever(kg, vector_store, linker, traversal_hops=2, alpha=0.5)

## 2.7 Graph RAG Demo

In [ ]:
# Entity-centric query
result = graph_rag.query("What did Elena Vasquez develop and what problem does it solve?")

print("\n" + "="*70 + "\nANSWER:\n" + "="*70)
print(result.answer)

In [ ]:
# Multi-hop query
result2 = graph_rag.query("Which organizations work on quantum training challenges?")

print("\n" + "="*70 + "\nANSWER:\n" + "="*70)
print(result2.answer)

In [ ]:
# Explore graph structure
print("Graph Exploration:")
print("="*50)

paths = kg.find_paths("elena vasquez", "barren plateau")
print(f"\nPaths (Elena Vasquez → barren plateau):")
for path in paths:
    print(f"  {' → '.join(path)}")

print(f"\nResearchers: {[e.name for e in kg.get_entities_by_type('PERSON')]}")
print(f"Algorithms: {[e.name for e in kg.get_entities_by_type('ALGORITHM')]}")

---

# 3) Comparison

| Scenario | Best Method | Why |
|----------|-------------|-----|
| Ambiguous query | **Agentic** | Iterative refinement clarifies intent |
| Entity-centric | **Graph** | Direct entity lookup |
| Multi-hop reasoning | **Graph** | Path traversal connects entities |
| Exploratory research | **Agentic** | Accumulates diverse context |
| Vocabulary mismatch | **Agentic** | Learns terminology from partial results |

In [ ]:
# Side-by-side comparison
test_query = "What solutions exist for variational quantum circuit training?"

print("="*70)
print(f"Query: {test_query}")
print("="*70)

print("\n--- AGENTIC RAG ---")
agentic_result = agentic_rag.query(test_query, verbose=False)
print(f"Iterations: {agentic_result.total_iterations}, Docs: {len(agentic_result.final_context)}")
print(f"Answer: {agentic_result.answer[:200]}...")

print("\n--- GRAPH RAG ---")
graph_result = graph_rag.query(test_query, verbose=False)
print(f"Entities: {len(graph_result.query_entities)}, Relations: {len(graph_result.traversed_relations)}")
print(f"Answer: {graph_result.answer[:200]}...")